# Day 14 · Pandas 进阶## 今日学习目标- 掌握布尔索引与多条件过滤- 掌握 groupby 分组聚合- 掌握 merge/concat 合并与透视

## 引入思考: `df.loc[1:3]` 和 `df.iloc[1:3]` 取的行数一样吗?- `loc` 是**标签**索引,包含结束位置(闭区间)- `iloc` 是**整数位置**索引,不含结束位置(左闭右开)今天重点学习:如何像 SQL 一样,用 Pandas 做过滤、分组、合并。

## 第一讲:过滤与排序

### 1.1 布尔索引

In [1]:
import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "年龄": [22, 28, 35, 42],    "城市": ["北京", "北京", "上海", "深圳"]})# 单条件过滤print("单条件: 年龄 > 30")print(df[df["年龄"] > 30])print()# 多条件过滤(每个条件必须加括号)print("多条件: 年龄 > 25 and 城市 == '北京'")print(df[(df["年龄"] > 25) & (df["城市"] == "北京")])

单条件: 年龄 > 30
   姓名  年龄 城市
2  王五  35 上海
3  赵六  42 深圳

多条件: 年龄 > 25 and 城市 == '北京'
   姓名  年龄 城市
1  李四  28 北京


> 易错点:多条件过滤必须加括号!写 `df[df["年龄"]>25 & df["城市"]=="北京"]` 会报错。每个条件要用 `()` 包裹,再用 `&`(且)/`|`(或)连接。

### 1.2 query() 方法

In [2]:
import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "年龄": [22, 28, 35, 42],    "城市": ["北京", "北京", "上海", "深圳"]})# 用字符串写条件,更贴近 SQL 风格result = df.query("年龄 > 25 and 城市 == '北京'")print("query 写法:")print(result)print()# 引用外部变量用 @ 符号min_age = 30print(f"引用变量 @min_age={min_age}:")print(df.query("年龄 >= @min_age"))

query 写法:
   姓名  年龄 城市
1  李四  28 北京

引用变量 @min_age=30:
   姓名  年龄 城市
2  王五  35 上海
3  赵六  42 深圳


### 1.3 排序

In [3]:
import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "年龄": [22, 28, 35, 42],    "城市": ["北京", "北京", "上海", "深圳"]})# 单列升序print("按年龄升序:")print(df.sort_values("年龄"))print()# 单列降序print("按年龄降序:")print(df.sort_values("年龄", ascending=False))print()# 多列排序print("按城市升序,再按年龄降序:")print(df.sort_values(    ["城市", "年龄"],    ascending=[True, False]))

按年龄升序:
   姓名  年龄 城市
0  张三  22 北京
1  李四  28 北京
2  王五  35 上海
3  赵六  42 深圳

按年龄降序:
   姓名  年龄 城市
3  赵六  42 深圳
2  王五  35 上海
1  李四  28 北京
0  张三  22 北京

按城市升序,再按年龄降序:
   姓名  年龄 城市
1  李四  28 北京
0  张三  22 北京
2  王五  35 上海
3  赵六  42 深圳


### ✏️ 练习 1

In [ ]:
import pandas as pdstu = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "成绩": [85, 92, 76, 88],    "班级": ["A", "A", "B", "A"]})# 学员代码区:用布尔索引筛选"成绩>=80 且 班级=='A'"filtered = None  # 在此作答pass

In [4]:
# 参考答案stu = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "成绩": [85, 92, 76, 88],    "班级": ["A", "A", "B", "A"]})filtered = stu[(stu["成绩"] >= 80) & (stu["班级"] == "A")]print("参考答案:")print(filtered)

参考答案:
   姓名 成绩 班级
0  张三  85  A
1  李四  92  A
3  赵六  88  A


In [ ]:
import pandas as pdstu = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "成绩": [85, 92, 76, 88],    "班级": ["A", "A", "B", "A"]})# 学员代码区:用 query() 重写条件,再按成绩降序result = None  # 在此作答pass

In [5]:
# 参考答案stu = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "成绩": [85, 92, 76, 88],    "班级": ["A", "A", "B", "A"]})result = stu.query("成绩 >= 80 and 班级 == 'A'") \\
           .sort_values("成绩", ascending=False)print("参考答案:")print(result)

参考答案:
   姓名 成绩 班级
1  李四  92  A
3  赵六  88  A
0  张三  85  A


## 第二讲:分组聚合

### 2.1 groupby

In [6]:
import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八"],    "成绩": [85, 92, 76, 88, 70, 79],    "班级": ["A", "A", "B", "A", "B", "B"]})# 按班级分组,求成绩均值avg = df.groupby("班级")["成绩"].mean()print("各班平均成绩:")print(avg)

各班平均成绩:
班级
A    88.333333
B    74.500000
Name: 成绩, dtype: float64


### 2.2 agg 多聚合函数

In [7]:
import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八"],    "成绩": [85, 92, 76, 88, 70, 79],    "班级": ["A", "A", "B", "A", "B", "B"]})# 一次调用多个聚合函数result = df.groupby("班级")["成绩"].agg(    ["mean", "max", "min", "count"])print("agg 多函数:")print(result)print()# 命名聚合:给每列起中文名named = df.groupby("班级")["成绩"].agg(    平均分="mean",    最高分="max",    最低分="min")print("命名聚合:")print(named)

agg 多函数:
     mean  max  min  count
班级                        
A   88.333333   92   85      3
B   74.500000   79   70      3

命名聚合:
     平均分  最高分  最低分
班级                    
A   88.333333    92    85
B   74.500000    79    70


### ✏️ 练习 2

In [ ]:
import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八"],    "成绩": [85, 92, 76, 88, 70, 79],    "班级": ["A", "A", "B", "A", "B", "B"]})# 学员代码区:按班级分组,# 计算平均分、最高分、最低分stats = Nonepass

In [8]:
# 参考答案import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八"],    "成绩": [85, 92, 76, 88, 70, 79],    "班级": ["A", "A", "B", "A", "B", "B"]})stats = df.groupby("班级")["成绩"].agg(    平均分="mean", 最高分="max", 最低分="min")print("参考答案:")print(stats)

参考答案:
     平均分  最高分  最低分
班级                    
A   88.333333    92    85
B   74.500000    79    70


In [ ]:
import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八"],    "成绩": [85, 92, 76, 88, 70, 79],    "班级": ["A", "A", "B", "A", "B", "B"],    "性别": ["男", "女", "男", "女", "男", "女"]})# 学员代码区:按性别和班级交叉分组,# 同时计算平均成绩cross = Nonepass

In [9]:
# 参考答案import pandas as pddf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八"],    "成绩": [85, 92, 76, 88, 70, 79],    "班级": ["A", "A", "B", "A", "B", "B"],    "性别": ["男", "女", "男", "女", "男", "女"]})cross = df.groupby(["性别", "班级"])["成绩"].mean()print("参考答案:")print(cross)

参考答案:
性别  班级
女   A     90.0
    B     79.0
男   A     85.0
    B     73.0
Name: 成绩, dtype: float64


## 第三讲:合并与透视

### 3.1 merge

In [10]:
import pandas as pdleft = pd.DataFrame({    "姓名": ["张三", "李四", "王五"],    "班级": ["A", "B", "A"]})right = pd.DataFrame({    "姓名": ["张三", "李四"],    "学号": [101, 102],    "数学": [90, 85]})# 内连接:只保留两边都有的键inner = pd.merge(left, right, on="姓名", how="inner")print("内连接 how='inner':")print(inner)print()# 左连接:保留左边全部,缺的填 NaNleft_join = pd.merge(left, right, on="姓名", how="left")print("左连接 how='left':")print(left_join)

内连接 how='inner':
   姓名 班级  学号  数学
0  张三  A  101  90
1  李四  B  102  85

左连接 how='left':
   姓名 班级    学号    数学
0  张三  A  101.0  90.0
1  李四  B  102.0  85.0
2  王五  A    NaN   NaN


### 3.2 concat

In [11]:
import pandas as pda = pd.DataFrame({"姓名": ["张三", "李四"], "年龄": [20, 21]})b = pd.DataFrame({"姓名": ["王五", "赵六"], "年龄": [22, 23]})# 纵向拼接:行叠加v = pd.concat([a, b], ignore_index=True)print("纵向拼接(默认 axis=0):")print(v)print()# 横向拼接:列对齐left = pd.DataFrame({"姓名": ["张三", "李四"], "年龄": [20, 21]})right = pd.DataFrame({"成绩": [85, 92], "等级": ["B", "A"]})h = pd.concat([left, right], axis=1)print("横向拼接(axis=1):")print(h)

纵向拼接(默认 axis=0):
   姓名  年龄
0  张三  20
1  李四  21
2  王五  22
3  赵六  23

横向拼接(axis=1):
   姓名  年龄  成绩 等级
0  张三  20   85  B
1  李四  21   92  A


### 3.3 pivot_table

In [12]:
import pandas as pdsales = pd.DataFrame({    "日期": ["5月1", "5月1", "5月2", "5月2"],    "城市": ["北京", "上海", "北京", "上海"],    "销量": [200, 300, 180, 150]})pivot = sales.pivot_table(    index="日期",    columns="城市",    values="销量",    aggfunc="sum")print("透视表: 日期 × 城市 = 销量")print(pivot)

透视表: 日期 × 城市 = 销量
城市   上海  北京
日期            
5月1  300  200
5月2  150  180


> 易错点:merge 后务必检查 `.shape`,确认行数是否符合预期。左连接后出现意料外的 NaN,通常是右表缺少对应键。

### ✏️ 练习 3

In [ ]:
import pandas as pdinfo = pd.DataFrame({    "学号": [101, 102, 103],    "姓名": ["张三", "李四", "王五"],    "班级": ["A", "B", "A"]})score = pd.DataFrame({    "学号": [101, 102],    "数学": [90, 85],    "英语": [88, 92]})# 学员代码区:分别用 inner / left 合并,# 观察两种连接的结果差异inner = Noneleft = Nonepass

In [13]:
# 参考答案import pandas as pdinfo = pd.DataFrame({    "学号": [101, 102, 103],    "姓名": ["张三", "李四", "王五"],    "班级": ["A", "B", "A"]})score = pd.DataFrame({    "学号": [101, 102],    "数学": [90, 85],    "英语": [88, 92]})inner = pd.merge(info, score, on="学号", how="inner")left  = pd.merge(info, score, on="学号", how="left")print("inner:", inner.shape)print(inner)print("\nleft:", left.shape)print(left)

inner: (2, 5)
   姓名 班级  学号  数学  英语
0  张三  A  101  90  88
1  李四  B  102  85  92

left: (3, 5)
   姓名 班级    学号    数学    英语
0  张三  A  101.0  90.0  88.0
1  李四  B  102.0  85.0  92.0
2  王五  A    NaN   NaN   NaN


## 第四讲:缺失值处理(补充)

### 4.1 检测与处理缺失值

In [14]:
import pandas as pdimport numpy as npdf = pd.DataFrame({    "姓名": ["张三", "李四", "王五"],    "年龄": [20, np.nan, 22],    "成绩": [85, 92, np.nan]})print("原始数据:")print(df)print()# 检测缺失值print("isnull 检测:")print(df.isnull())print()# 填充 0print("填充 0:")print(df.fillna(0))print()# 列均值填充print("列均值填充:")print(df.fillna(df.mean(numeric_only=True)))print()# 前向填充print("前向填充:")print(df.ffill())

原始数据:
   姓名    年龄   成绩
0  张三  20.0  85.0
1  李四   NaN  92.0
2  王五  22.0   NaN

isnull 检测:
      姓名     年龄     成绩
0  False  False  False
1  False   True  False
2  False  False   True

填充 0:
   姓名    年龄   成绩
0  张三  20.0  85.0
1  李四   0.0  92.0
2  王五  22.0   0.0

列均值填充:
   姓名    年龄   成绩
0  张三  20.0  85.0
1  李四  21.0  92.0
2  王五  22.0  88.5

前向填充:
   姓名    年龄   成绩
0  张三  20.0  85.0
1  李四  20.0  92.0
2  王五  22.0  92.0


### ✏️ 练习 4 (缺失值)

In [ ]:
import pandas as pdimport numpy as npdf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "年龄": [20, np.nan, 22, np.nan],    "成绩": [85, 92, np.nan, 78]})# 学员代码区:# 1) 检测缺失值# 2) 分别用 0 / 列均值 / 前向填充 三种策略处理check = Nonefill_zero = Nonefill_mean = Nonefill_ffill = Nonepass

In [15]:
# 参考答案import pandas as pdimport numpy as npdf = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "年龄": [20, np.nan, 22, np.nan],    "成绩": [85, 92, np.nan, 78]})print("检测结果:")print(df.isnull())print()print("填充 0:")print(df.fillna(0))print()print("列均值填充:")print(df.fillna(df.mean(numeric_only=True)))print()print("前向填充:")print(df.ffill())

检测结果:
      姓名     年龄     成绩
0  False  False  False
1  False   True  False
2  False  False   True
3  False   True  False

填充 0:
   姓名    年龄   成绩
0  张三  20.0  85.0
1  李四   0.0  92.0
2  王五  22.0   0.0
3  赵六   0.0  78.0

列均值填充:
   姓名    年龄   成绩
0  张三  20.0  85.0
1  李四  21.0  92.0
2  王五  22.0  85.0
3  赵六  21.0  78.0

前向填充:
   姓名    年龄   成绩
0  张三  20.0  85.0
1  李四  20.0  92.0
2  王五  22.0  92.0
3  赵六  22.0  78.0


## 总结| 操作 | 方法 | 一句话 ||---|---|---|| 单/多条件过滤 | `df[条件]` / `df.query()` | 多条件必须加括号 || 排序 | `df.sort_values()` | `ascending` 控制升降 || 分组聚合 | `df.groupby().agg()` | 命名聚合更直观 || 合并 | `pd.merge()` | inner/left/right/outer || 拼接 | `pd.concat()` | axis=0 纵向,1 横向 || 透视 | `df.pivot_table()` | 适合交叉汇总 || 缺失值 | `fillna` / `dropna` / `ffill` | 先检测 `isnull` |

## ⭐ 挑战题

In [ ]:
import pandas as pdscore = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],    "班级": ["A", "A", "B", "B", "A"],    "成绩": [85, 55, 72, 90, 63]})class_info = pd.DataFrame({    "班级": ["A", "B"],    "班主任": ["王老师", "李老师"],    "教室": ["101", "102"]})# 学员代码区:综合场景——# 1) 过滤成绩 > 60# 2) 按班级分组统计(平均、最高、最低)# 3) 再与班级信息表 mergeresult = Nonepass

In [17]:
# 参考答案import pandas as pdscore = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],    "班级": ["A", "A", "B", "B", "A"],    "成绩": [85, 55, 72, 90, 63]})class_info = pd.DataFrame({    "班级": ["A", "B"],    "班主任": ["王老师", "李老师"],    "教室": ["101", "102"]})# 1) 过滤filtered = score[score["成绩"] > 60]# 2) 分组统计stats = filtered.groupby("班级")["成绩"].agg(    平均="mean", 最高="max", 最低="min").reset_index()# 3) mergeresult = pd.merge(stats, class_info, on="班级")print("参考答案:")print(result)

参考答案:
  班级         平均  最高  最低  班主任  教室
0  A  74.000000  85  63  王老师  101
1  B  81.000000  90  72  李老师  102
